In [48]:
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import boto3
import json
import gzip
from datetime import datetime
from io import StringIO, BytesIO
import requests
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [46]:
s3_client = boto3.client("s3")

bucket = "s3-3601-final-project"
prefix = "ncdot/traffic/"

url = "https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NCDOT_AADT_Stations/FeatureServer/0/query"

location_ids = [
    "0600000063",
    "0600000305",
    "0600000145",
    "0600000829",
    "0600000107",
    "0600000108",
    "0600000414",
    "0600000412",
    "0600000044",
    "0600000065",
    "0600001182"
]
# Construct the where clause for the API request based on the location IDs
where = "LocationID IN ({})".format(",".join(f"'{x}'" for x in location_ids))
params = {
    "where": where,
    "outFields": "LocationID,COUNTY,ROUTE,LOCATION,AADT_2022",
    "returnGeometry": "true",
    "outSR": '{"wkid":4326}',
    "f": "json"
}
# Send the API request to fetch the traffic data for the specified location IDs
response = requests.get(url, params=params)
response.raise_for_status()
#
api_data = response.json()
#checks that the coordinates were updated correctly
print(api_data["features"][0])

ncdot_df = pd.json_normalize(api_data["features"])

#renaming columns
ncdot_df = ncdot_df.rename(columns={
    "attributes.LocationID": "LocationID",
    "attributes.COUNTY": "County",
    "attributes.ROUTE": "Route",
    "attributes.LOCATION": "Location",
    "attributes.AADT_2022": "AADT_2022",
    "geometry.x": "longitude",
    "geometry.y": "latitude"
})

print(ncdot_df)


{'attributes': {'LocationID': '0600000044', 'COUNTY': 'MECKLENBURG', 'ROUTE': 'US 29/NC 49', 'LOCATION': 'EAST OF STETSON DR', 'AADT_2022': '29500'}, 'geometry': {'x': -80.75957420178952, 'y': 35.28798544117831}}
    LocationID       County                     Route  \
0   0600000044  MECKLENBURG               US 29/NC 49   
1   0600000063  MECKLENBURG                     NC 49   
2   0600000065  MECKLENBURG                     US 29   
3   0600000107  MECKLENBURG                     NC 49   
4   0600000108  MECKLENBURG                     NC 49   
5   0600000145  MECKLENBURG                     NC 49   
6   0600000305  MECKLENBURG                     US 29   
7   0600000412  MECKLENBURG    NC 24 (WT HARRIS BLVD)   
8   0600000414  MECKLENBURG  NC 24 (E WT HARRIS BLVD)   
9   0600000829  MECKLENBURG                     NC 49   
10  0600001182  MECKLENBURG                   SR 2833   

                            Location AADT_2022  longitude   latitude  
0                 EAST OF STETS

In [47]:
# Include date and time
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")
file_key = f"ncdot/traffic/aadt_stations_{timestamp}.csv"

# Compress the JSON data
csv_string = ncdot_df.to_csv(index=False)

# Upload to S3
s3_client.put_object(
    Bucket=bucket,
    Key=file_key,
    Body=csv_string.encode("utf-8"),
    ContentType="text/csv",
)

print("Saved file:")
print(f"s3://{bucket}/{file_key}")

Saved file:
s3://s3-3601-final-project/tomtom/traffic/aadt_stations_2026-09-27_15-13-28-302554.csv


In [70]:
#I could be wrong but i think we can delete this? Copy it into a google doc in case we need it later. 

response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)

data_list = []
# Iterate through the list of objects in the S3 bucket and 
# process each .json.gz file
for obj in response.get("Contents", []):

    current_file_key = obj["Key"]

    # Only read .json.gz files
    if not current_file_key.endswith(".json.gz"):
        continue

    print("Reading:", current_file_key)

    # Get the file from S3
    file_response = s3_client.get_object(Bucket=bucket, Key=current_file_key)

    compressed_content = file_response["Body"].read()

    # Decompress file
    with gzip.GzipFile(fileobj=BytesIO(compressed_content)) as gzip_file:
        raw_json_data = json.loads(gzip_file.read().decode("utf-8"))

    # Convert features to DataFrame
    features = raw_json_data.get("features")

    if not isinstance(features, list):
        print("Skipping file without features:", current_file_key)
        print("Response:", raw_json_data.get("error", raw_json_data))
        continue

    temp_df = pd.json_normalize(features)

    # Remove "attributes." from column names
    temp_df.columns = [
        col.replace("attributes.", "")
        for col in temp_df.columns
    ]

    # Keep track of which S3 file the data came from
    temp_df["source_file"] = current_file_key
    data_list.append(temp_df)

Reading: ncdot/traffic/aadt_stations.json.gz
Reading: ncdot/traffic/aadt_stations_2026-09-26_13-59-22.json.gz
Reading: ncdot/traffic/aadt_stations_2026-09-27_03-28-24-080898.json.gz
Reading: ncdot/traffic/aadt_stations_2026-09-27_03-38-13-507221.json.gz
Reading: ncdot/traffic/aadt_stations_2026-09-27_03-45-18-289308.json.gz
Reading: ncdot/traffic/aadt_stations_2026-09-27_04-09-04-838062.json.gz
Skipping file without features: ncdot/traffic/aadt_stations_2026-09-27_04-09-04-838062.json.gz
Response: {'code': 400, 'message': '', 'details': ["'outSR' parameter is invalid"]}
Reading: ncdot/traffic/aadt_stations_2026-09-27_04-14-08-062826.json.gz
Skipping file without features: ncdot/traffic/aadt_stations_2026-09-27_04-14-08-062826.json.gz
Response: {'code': 400, 'message': '', 'details': ["'outSR' parameter is invalid"]}
Reading: ncdot/traffic/aadt_stations_2026-09-27_04-14-56-669941.json.gz
Skipping file without features: ncdot/traffic/aadt_stations_2026-09-27_04-14-56-669941.json.gz
Respo

In [71]:
# Same as above, put in google doc and remove
#Combines S3 files
if data_list: ncdot_df = pd.concat(data_list, ignore_index=True)

else:
    ncdot_df = pd.DataFrame()

print("\nCombined NCDOT data:")
print(ncdot_df.head(11))


Combined NCDOT data:
    LocationID                     ROUTE                          LOCATION  \
0   0600000063                     NC 49                     WEST OF I-485   
1   0600000145                     NC 49  EAST OF SR 5394 MALLARD CK CH RD   
2   0600000829                     NC 49                   EAST OF SR 2939   
3   0600000107                     NC 49                 EAST OF SUTHER RD   
4   0600000108                     NC 49                 WEST OF SUTHER RD   
5   0600000414  NC 24 (E WT HARRIS BLVD)                   EAST OF SR 5316   
6   0600000412    NC 24 (WT HARRIS BLVD)                    SOUTH OF US 29   
7   0600000044               US 29/NC 49                EAST OF STETSON DR   
8   0600000065                     US 29   SOUTH OF UNIVERSITY POINTE BLVD   
9   0600001182                   SR 2833                    NORTH OF NC 49   
10  0600000044               US 29/NC 49                EAST OF STETSON DR   

   AADT_2022    geometry.x     geometry.y